# Profiling automatizado — Olist

Este notebook cria uma base analítica com uma linha por pedido e gera um relatório HTML com `fg-data-profiling`. Os CSVs brutos não são alterados.

In [1]:
from pathlib import Path

import pandas as pd
from data_profiling import ProfileReport

PROJECT_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
RAW_DIR = PROJECT_DIR / 'data' / 'raw'
REPORT_DIR = PROJECT_DIR / 'reports'
REPORT_DIR.mkdir(exist_ok=True)
REPORT_PATH = REPORT_DIR / 'olist_order_level_profile.html'
RAW_DIR, REPORT_PATH

c:\Users\1t1.847986\Documents\ml_2026\olist-project\olist\.profiling-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(WindowsPath('C:/Users/1t1.847986/Documents/ml_2026/olist-project/olist/data/raw'),
 WindowsPath('C:/Users/1t1.847986/Documents/ml_2026/olist-project/olist/reports/olist_order_level_profile.html'))

In [2]:
orders = pd.read_csv(RAW_DIR / 'olist_orders_dataset.csv', parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date',
])
customers = pd.read_csv(RAW_DIR / 'olist_customers_dataset.csv')
items = pd.read_csv(RAW_DIR / 'olist_order_items_dataset.csv')
payments = pd.read_csv(RAW_DIR / 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(RAW_DIR / 'olist_order_reviews_dataset.csv')

orders.shape, customers.shape, items.shape, payments.shape, reviews.shape

((99441, 8), (99441, 5), (112650, 7), (103886, 5), (99224, 7))

In [3]:
# Agregações preservam uma única linha por pedido e evitam duplicações nos joins.
item_summary = items.groupby('order_id', as_index=False).agg(
    item_count=('order_item_id', 'count'),
    product_count=('product_id', 'nunique'),
    seller_count=('seller_id', 'nunique'),
    product_value=('price', 'sum'),
    freight_value=('freight_value', 'sum'),
)
payment_summary = payments.groupby('order_id', as_index=False).agg(
    payment_count=('payment_sequential', 'count'),
    payment_value=('payment_value', 'sum'),
    max_installments=('payment_installments', 'max'),
)
review_summary = reviews.groupby('order_id', as_index=False).agg(
    review_count=('review_id', 'count'),
    review_score_mean=('review_score', 'mean'),
)

profile_data = (
    orders
    .merge(customers, on='customer_id', how='left')
    .merge(item_summary, on='order_id', how='left')
    .merge(payment_summary, on='order_id', how='left')
    .merge(review_summary, on='order_id', how='left')
)
profile_data['delivery_days'] = (
    profile_data['order_delivered_customer_date'] - profile_data['order_purchase_timestamp']
).dt.total_seconds() / 86_400
profile_data['delivery_vs_estimate_days'] = (
    profile_data['order_delivered_customer_date'] - profile_data['order_estimated_delivery_date']
).dt.total_seconds() / 86_400
profile_data.shape

(99441, 24)

In [4]:
# IDs são mantidos como texto, mas removidos do perfil para evitar análises pouco úteis de alta cardinalidade.
report_data = profile_data.drop(columns=['order_id', 'customer_id', 'customer_unique_id'])
report = ProfileReport(
    report_data,
    title='Olist — Perfil da base analítica por pedido',
    explorative=True,
    minimal=False,
)
report.to_file(REPORT_PATH)
REPORT_PATH

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 39.87it/s]


WindowsPath('C:/Users/1t1.847986/Documents/ml_2026/olist-project/olist/reports/olist_order_level_profile.html')